# UK Postcode Lookup - postcodes.io

Demonstrates `lookup_postcode()` and `lookup_outcode()` from the `airgap_geo`
library, which query the self-hosted **postcodes.io** API for UK postcode and
outcode (district) metadata.

## Prerequisites

| Service          | Default port | Role                       |
| ---------------- | ------------ | -------------------------- |
| postcodes.io API | 8000         | UK postcode / outcode data |


______________________________________________________________________

## Start Services

Before running this notebook, make sure the required Docker services are up.
From the **project root**, run:

```bash
make postcodes-up        # starts postcodes.io API + database
```

Check running containers with `make ps`. See the main
[README](../README.md#docker-services) for all available profiles and
configuration details.


## Setup - Imports and Configuration


In [ ]:
import importlib
import pprint

import httpx
import pandas as pd
import requests

import airgap_geo.settings as _settings

importlib.reload(_settings)

from airgap_geo import lookup_outcode, lookup_postcode
from airgap_geo.settings import POSTCODES_URL

client = httpx.AsyncClient()

print(f"postcodes.io : {POSTCODES_URL}")

## Health Check


In [ ]:
try:
    r = requests.get(POSTCODES_URL, timeout=5)
    print(f"postcodes.io  \u2713  HTTP {r.status_code}  ({POSTCODES_URL})")
except Exception:
    print(f"postcodes.io  \u2717  unreachable  ({POSTCODES_URL})")

______________________________________________________________________

## 1 - Full Postcode Lookup

`lookup_postcode(postcode)` returns the full metadata for a UK postcode, including
coordinates, administrative geography, parliamentary constituency, and more.


In [ ]:
pc_result = await lookup_postcode("SW1A 2AA", client)
pprint.pprint(pc_result)

______________________________________________________________________

## 2 - Outcode (District) Lookup

`lookup_outcode(outcode)` returns aggregated metadata for an outward code (the
first part of a UK postcode, e.g. `"SW1A"`).


In [ ]:
oc_result = await lookup_outcode("SW1A", client)
pprint.pprint(oc_result)

______________________________________________________________________

## 3 - Comparison Table


In [ ]:
_PC_FIELDS = [
    "postcode",
    "latitude",
    "longitude",
    "admin_district",
    "admin_ward",
    "region",
    "country",
    "nuts",
]

rows = []
for field in _PC_FIELDS:
    rows.append(
        {
            "Field": field,
            "SW1A 2AA": getattr(pc_result, field, "-") if pc_result else "-",
            "SW1A (outcode)": getattr(oc_result, field, "-") if oc_result else "-",
        }
    )

pd.DataFrame(rows).set_index("Field")

______________________________________________________________________

## 4 - Error Handling

Invalid postcodes and outcodes return an empty dict rather than raising exceptions.


In [ ]:
bad_pc = await lookup_postcode("ZZ9 9ZZ", client)
print(f"Invalid postcode  \u2192 {bad_pc!r}  (expected: {{}})")

bad_oc = await lookup_outcode("ZZ9", client)
print(f"Invalid outcode   \u2192 {bad_oc!r}  (expected: {{}})")

______________________________________________________________________

## Teardown - Close HTTP Client


In [ ]:
await client.aclose()

______________________________________________________________________

## Stop Services

To stop containers, use `make down` (or a profile-specific target such as
`make postcodes-down`) from the project root. See the
[README](../README.md#docker-services) for details.
